# Generate all answer speakers

Run all cells to create the Kokoro speakers for existing complete answer sets. Heart remains the default and its files are never overwritten. Bella speaks the question previews. Other voices unlock in website Settings only after their complete recordings are ready. Downloads and packages stay in `.cache/`. This job is resumable.

Model: https://huggingface.co/hexgrad/Kokoro-82M
Runtime: https://github.com/thewh1teagle/kokoro-onnx

In [ ]:
from pathlib import Path
import os, sys, subprocess, venv
project = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'bin' / 'generate_all_voices.py').exists())
os.chdir(project)
environment = project / '.cache' / 'answer-voices-env'
python = environment / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
if not python.exists():
    venv.EnvBuilder(with_pip=True, system_site_packages=True).create(environment)
subprocess.run([str(python), '-m', 'pip', 'install', 'kokoro-onnx', 'soundfile', 'espeakng-loader'], check=True)
print('Isolated Kokoro runtime ready:', python)


In [ ]:
# None means every voice included in Kokoro's voice pack (all 54 speakers).
# The English scripts stay English; non-English voices are used as alternate timbres.
VOICES = None
QUESTION_VOICE = 'af_bella'  # Separate from af_heart, the default answer voice.


In [ ]:
import json
log_path = project / '.cache' / 'answer-voice-generation.log'
command = "import sys; sys.path.insert(0, 'bin'); from generate_all_voices import run; run(voices=" + repr(VOICES) + ", question_voice=" + repr(QUESTION_VOICE) + ")"
print('Generation started. Progress and errors appear below and in:', log_path, flush=True)
print('Existing Heart recordings are preserved. Rerun to resume interrupted work.', flush=True)
environment_variables = dict(os.environ, PYTHONIOENCODING='utf-8')
with log_path.open('a', encoding='utf-8') as log:
    with subprocess.Popen([str(python), '-u', '-c', command], cwd=project,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', env=environment_variables) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
if return_code:
    raise RuntimeError(f'Voice generation exited with code {return_code}. The actual error is printed above. Full log: {log_path}')
print('Finished. Refresh the website to choose another answer speaker under Settings.')
